# 转换脚本

In [ ]:
import nbformat

def export_cells_to_script(notebook_path, cell_indices, script_path):
    # 读取 notebook 文件
    with open(notebook_path) as f:
        nb = nbformat.read(f, as_version=4)
    
    # 打开文件以写入代码
    with open(script_path, 'w') as script_file:        
        script_file.write('''
import argparse

parser = argparse.ArgumentParser(description="Example script with command-line arguments.")

parser.add_argument('--file_path', type=str, required=True)
parser.add_argument('--origin_video_path', type=str, required=True)
parser.add_argument('--yolo_video_path', type=str, required=True)
parser.add_argument('--labels_path', type=str, required=True)

args = parser.parse_args()
file_path = args.file_path
origin_video_path = args.origin_video_path
yolo_video_path = args.yolo_video_path  
labels_path = args.labels_path  
                      
''')
        # 遍历所有指定的 cell 索引
        for cell_index in cell_indices:
            # 获取指定 cell
            cell = nb['cells'][cell_index]
            
            # 确保 cell 是 code 类型
            if cell['cell_type'] == 'code':
                code = cell['source']
                script_file.write(f"# Cell {cell_index}\n")
                script_file.write(code)
                script_file.write("\n\n")  # 添加空行分隔不同的 cell
            # else:
                # print(f"Cell {cell_index} is not a code cell.")

    print(f"Cells {cell_indices} exported to {script_path}")


export_cells_to_script('frame_extract_sample.ipynb', [i for i in range(3, 29)], 'frame_extract_script.py')

# python frame_extract_script.py 
# --file_path yolo-video/video_liaixiang 
# --origin_video_path yolo-video/video_liaixiang/shz032.avi 
# --yolo_video_path yolo-video/video_liaixiang/shz032.mp4 -
# -labels_path yolo-video/video_liaixiang/labels

# 处理数据集中的所有视频

In [ ]:
import os, shutil
import subprocess
from matplotlib import pyplot as plt
from tqdm import tqdm
import time


exp_path = "./exp/qinzhou1"
data_path = "./data/qinzhouVideo/TICVideos"
weight = "yolov5s-epoch500-batch16-fpn+ghost"

video_dir_names = os.listdir(data_path)
video_dir_path = [os.path.join(data_path, video_name) for video_name in video_dir_names]

# 判断是否是文件夹，有的是系统隐藏文件过滤掉
for index, dir in enumerate(video_dir_path):
    if not os.path.isdir(dir):
        print(f'Remove [{dir}]')
        del video_dir_path[index]
        del video_dir_names[index]

video_dir_path.sort()
video_dir_names.sort()

pbar = tqdm(total=len(video_dir_path), desc="PROCESSING", ncols=None)

# 处理每个视频  video_dir_path：原视频路径
for index, dir in enumerate(video_dir_path):
    pbar.set_description_str(f"PROCESSING {video_dir_names[index]}")

    video_path = ''                                              # 源视频路径
    file_path = os.path.join(exp_path, video_dir_names[index])   # 保存yolo结果及分析的文件夹路径，exp目录下

    for file in os.listdir(dir):
        if file.endswith(('.avi', '.wmv')):
            # video_path = os.path.join(dir, file)
            video_path = os.path.join(dir, f"{video_dir_names[index]}_cut.mp4")

    # 视频文件名
    file_name = os.path.basename(video_path)
    file_name = os.path.splitext(file_name)[0]

    # 没有保存过yolo结果mp4，则进行yolo处理，此前删除保存不完整yolo结果的目录
    time_yolo_phase = time.time()
    if not os.path.exists(os.path.join(file_path, video_dir_names[index]+'_yolo.mp4')):  

        # 删除保存不完整yolo结果的目录
        if os.path.exists(file_path):
            print(f'[Warning] Delete incomplete yolo dir: {file_path}')
            shutil.rmtree(file_path)

        # print(f"YOLO process file: {video_path}")
        pbar.set_description_str(f"PROCESSING {video_dir_names[index]} [YOLO PHASE]")

        yolo_command = [
            "python", "../yolov5/segment/predict.py",
            "--img", "960",
            "--weights", "../yolov5/runs/train-seg/" + weight + "/weights/best.pt",
            "--source", video_path,
            "--save-txt",
            "--save-conf",
            "--retina-masks",
            "--project", exp_path,
            "--name", video_dir_names[index],
            "--device", "1"
        ]
        # print(" ".join(yolo_command))
        yolo_output = subprocess.run(yolo_command, capture_output=True, text=True, check=True)

        if yolo_output.returncode != 0:
            print(f"[Error] YOLO process file: {video_path} failed with returncode: {yolo_output.returncode}\n", yolo_output.stderr)

        # yolo输出在stderr中
        # print("\n")
        # for line in yolo_output.stderr.splitlines():
        #     if line.startswith('video') or len(line) == 0:
        #         continue
        #     print(line)
        # print("\n")

        with open(os.path.join(file_path, 'yolo_log.txt'), 'w') as f:
            f.write(yolo_output.stderr)
            
        # 重命名yolo生成的name_cut.mp4视频为name_yolo.mp4
        os.rename(os.path.join(file_path, f"{video_dir_names[index]}_cut.mp4"), os.path.join(file_path, f"{video_dir_names[index]}_yolo.mp4"))

    time_yolo_phase = time.time() - time_yolo_phase
    
    # 没有保存过分析结果metrics.txt，则进行分析
    time_analysis_phase = time.time()
    if not os.path.exists(os.path.join(file_path, 'metrics.txt')):  
        # yolo处理完后使用自己脚本分析结果
        pbar.set_description_str(f"PROCESSING {video_dir_names[index]} [ANALYSIS PHASE]")

        origin_video_path = video_path
        yolo_video_path = os.path.join(file_path, video_dir_names[index] + '_yolo.mp4')
        labels_path = os.path.join(file_path, 'labels')

        analysis_script_command = [
            'python', 'frame_extract_script.py',
            '--file_path', file_path,
            '--origin_video_path', origin_video_path,
            '--yolo_video_path', yolo_video_path,
            '--labels_path', labels_path
        ]
        analysis_script_output = subprocess.run(analysis_script_command, capture_output=True, text=True, check=True)

        if analysis_script_output.stdout:
            with open(os.path.join(file_path, 'analysis_log.txt'), 'w') as f:
                f.write(analysis_script_output.stdout)

        # 运行失败
        if analysis_script_output.returncode != 0:
            print(f"[Error] analysis_script error with returncode: {analysis_script_output.returncode}\n", analysis_script_output.stderr)
            break
    
    time_analysis_phase = time.time() - time_analysis_phase
    
    with open(os.path.join(file_path, 'time_consuming.txt'), 'a') as f:
        f.write(f"{dir} {time_yolo_phase:.2f} {time_analysis_phase:.2f}\n")
        
    pbar.update(1)
    

# 计算两张图的ssim psnr

In [ ]:
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
from PIL import Image
import numpy as np
import re

def get_rect_area(frame: Image.Image, rect: tuple):
    x, y, w, h = rect
    return frame.crop((x, y, x + w, y + h))

def get_area_from_log(analysis_log):
    rect_ultrasound = tuple()
    rect_contrast = tuple()

    # ()是分组，需要转义，\d+表示一个或多个数字
    with open(analysis_log, 'r') as f:
        for index, line in enumerate(f):
            if 10 < index < 18:
                pattern = r'\((\d+), (\d+), (\d+), (\d+)\)'
                matches = re.findall(pattern, line)
                if len(matches) == 2:
                    rects = [(int(x), int(y), int(w), int(h)) for x, y, w, h in matches]
                    rect_ultrasound = rects[0]
                    rect_contrast = rects[1]
                         
    return rect_ultrasound, rect_contrast

def calculate_ssim(image_path1, image_path2, area="ultrsound"):
    img1 = Image.open(image_path1).convert('RGB')
    img2 = Image.open(image_path2).convert('RGB')
    
    common_path = os.path.commonpath([image_path1, image_path2])
    analysis_log_path = os.path.join(os.path.dirname(common_path), 'analysis_log.txt')
    rect_ultrasound, rect_contrast = get_area_from_log(analysis_log_path)
    
    if area == "ultrsound":
        img1 = get_rect_area(img1, rect_ultrasound)
        img2 = get_rect_area(img2, rect_ultrasound)
    elif area == "contrast":
        img1 = get_rect_area(img1, rect_contrast)
        img2 = get_rect_area(img2, rect_contrast)
    
    img1_array = np.array(img1)
    img2_array = np.array(img2)
    
    ssim_values = []
    for i in range(3):  # 计算每个颜色通道的 SSIM
        ssim_value = ssim(img1_array[:, :, i], img2_array[:, :, i], data_range=img2_array[:, :, i].max() - img1_array[:, :, i].min())
        ssim_values.append(ssim_value)
    
    return np.mean(ssim_values)

def calculate_psnr(image_path1, image_path2, area="ultrsound"):
    img1 = Image.open(image_path1).convert('RGB')
    img2 = Image.open(image_path2).convert('RGB')
    
    common_path = os.path.commonpath([image_path1, image_path2])
    analysis_log_path = os.path.join(os.path.dirname(common_path), 'analysis_log.txt')
    rect_ultrasound, rect_contrast = get_area_from_log(analysis_log_path)
    
    if area == "ultrsound":
        img1 = get_rect_area(img1, rect_ultrasound)
        img2 = get_rect_area(img2, rect_ultrasound)
    elif area == "contrast":
        img1 = get_rect_area(img1, rect_contrast)
        img2 = get_rect_area(img2, rect_contrast)
    
    img1_array = np.array(img1)
    img2_array = np.array(img2)
    
    psnr_values = []
    for i in range(3):  # 计算每个颜色通道的 PSNR
        psnr_value = psnr(img1_array[:, :, i], img2_array[:, :, i], data_range=img2_array[:, :, i].max() - img1_array[:, :, i].min())
        psnr_values.append(psnr_value)
    
    return np.mean(psnr_values)

# 示例使用
path1 = './exp/hunan/0458/TTP/roi1_TTP.jpg'
path2 = './exp/hunan/0458/TTP/target_TTP.jpg'
ssim_result = calculate_ssim(path1, path2, area="")
psnr_result = calculate_psnr(path1, path2, area="")

print(f'SSIM: {ssim_result}')
print(f'PSNR: {psnr_result}')


# 查看TTP (所有)

In [ ]:
import os
import numpy as np
from random import choices

exp_path = "exp/qinzhou1"

save_path = "./exp_result"

exps = [
    os.path.join(exp_path, exp)
    for exp in os.listdir(exp_path)
    if os.path.isdir(os.path.join(exp_path, exp))
]
print(exps)
print(f'Total experiments: {len(exps)}')

x = []
patients = []
fps = []
frame_count = []
resolution = []
duration = []
medium_confidence = []
average_confidence = []
lost_frame_ratio = []
lost_frame_second = []
time_yolo_phase = []
time_analysis_phase = []
time_total = []
roi1_TTP = []
roi2_TTP = []
target_TTP = []
target_TTP_diff1 = []
target_TTP_diff2 = []
diff_TTP = []
diff_TTP_diff1 = []
diff_TTP_diff2 = []
contrast_TTP = []
contrast_TTP_diff1 = []
contrast_TTP_diff2 = []
roi1_peak_intensity = []
target_peak_intensity = []
ssim_list = []
psnr_list = []

duration_less_list = []

for exp in exps: 
    patient_name = os.path.basename(exp)
    
    txt = os.path.join(exp, "metrics.txt")
    if os.path.exists(txt):
        with open(txt, "r") as f:
            diff = f.read().split("\n")
            
            roi_img = os.path.join(exp, "TTP/roi1_TTP.jpg")
            target_img = os.path.join(exp, "TTP/target_TTP.jpg")
            ssim_list.append(calculate_ssim(roi_img, target_img))
            psnr_list.append(calculate_psnr(roi_img, target_img))
            patients.append(patient_name)
            fps.append(float(diff[0].split(" ")[1]))
            frame_count.append(float(diff[1].split(" ")[1]))
            resolution.append(diff[2].split(" ")[1])
            duration.append(float(diff[3].split(" ")[1]))
            medium_confidence.append(float(diff[4].split(" ")[1]))
            average_confidence.append(float(diff[5].split(" ")[1]))
            lost_frame_ratio.append(float(diff[6].split(" ")[1]))
            lost_frame_second.append(float(diff[7].split(" ")[1]))
            roi1_TTP.append(float(diff[8].split(" ")[1]))
            roi2_TTP.append(float(diff[9].split(" ")[1]))
            target_TTP.append(float(diff[10].split(" ")[1]))
            diff_TTP.append(float(diff[11].split(" ")[1]))
            contrast_TTP.append(float(diff[12].split(" ")[1]))
            
            target_TTP_diff1.append(abs(target_TTP[-1] - roi1_TTP[-1]))
            target_TTP_diff2.append(abs(target_TTP[-1] - roi2_TTP[-1]))
            
            diff_TTP_diff1.append(abs(diff_TTP[-1] - roi1_TTP[-1]))
            diff_TTP_diff2.append(abs(diff_TTP[-1] - roi2_TTP[-1]))
            
            contrast_TTP_diff1.append(abs(contrast_TTP[-1] - roi1_TTP[-1]))
            contrast_TTP_diff2.append(abs(contrast_TTP[-1] - roi2_TTP[-1]))
    # 没有metrics.txt文件，exps列表中删除这个exp
    else:
        print(f"No metrics.txt file in exp: {exp}. It can be no detections or havn't been analyzed yet.")
        exps.remove(exp)
            
    time_txt = os.path.join(exp, "time_consuming.txt")
    if os.path.exists(time_txt):
        with open(time_txt, "r") as f:
            line = f.readlines()[0].split()
            time_yolo_phase.append(float(line[1]))
            time_analysis_phase.append(float(line[2]))
            time_total.append(float(line[1]) + float(line[2]))
    else:
        print(f"No time_consuming.txt file in exp: {exp}. It can be no detections or havn't been analyzed yet.")
        exps.remove(exp)
        
    peak_intensity_txt = os.path.join(exp, "peak_intensity.txt")
    if os.path.exists(peak_intensity_txt):
        with open(peak_intensity_txt, "r") as f:
            lines = f.readlines()
            line0 = lines[0].split()
            line1 = lines[1].split()
            if line0[0].startswith('roi1'):
                roi1_peak_intensity.append(float(line0[1]))
            if line1[0].startswith('target'):
                target_peak_intensity.append(float(line1[1]))
    else:
        print(f"No peak_intensity.txt file in exp: {exp}. It can be no detections or havn't been analyzed yet.")
        exps.remove(exp)
    
print(f"duration_less_list: {len(duration_less_list)} {duration_less_list}")
x = range(len(roi1_TTP))

print("fps: {}\nduration: {:.2f} ~ {:.2f}".format(set(fps), min(duration), max(duration)))

threshould = 2

threshould_target = threshould
target_TTP_diff1_percent = len([x for x in target_TTP_diff1 if x < threshould_target]) / len(target_TTP_diff1)
target_TTP_diff2_percent = len([x for x in target_TTP_diff2 if x < threshould_target]) / len(target_TTP_diff2)

threshould_diff = threshould
diff_TTP_diff1_percent = len([x for x in diff_TTP_diff1 if x < threshould_diff]) / len(diff_TTP_diff1)
diff_TTP_diff2_percent = len([x for x in diff_TTP_diff2 if x < threshould_diff]) / len(diff_TTP_diff2)

threshould_contrast = threshould
contrast_TTP_diff1_percent = len([x for x in contrast_TTP_diff1 if x < threshould_contrast]) / len(contrast_TTP_diff1)
contrast_TTP_diff2_percent = len([x for x in contrast_TTP_diff2 if x < threshould_contrast]) / len(contrast_TTP_diff2)

fig, ax = plt.subplots(3, 4, figsize=(15, 15))

ax[0][2].set_title('Lost frame ratio')
ax[0][2].scatter(x, lost_frame_ratio, c="g", s=1, label="lost_frame_ratio")

ax[0][3].set_title('Duration')
ax[0][3].scatter(x, duration, c="b", s=4, label="duration")
ax[0][3].scatter(x, lost_frame_second, c="r", s=4, label="lost_frame_second")
ax[0][3].legend()

ax[1][2].set_title('frame_count')
ax[1][2].scatter(x, frame_count, c="r", s=4, label="frame_count")

ax[1][3].set_title('confidence')
ax[1][3].scatter(x, medium_confidence, c="g", s=4, label="medium_confidence")
ax[1][3].scatter(x, average_confidence, c="r", s=4, label="average_confidence")
ax[1][3].legend()

ax[2][2].set_title('roi TTP')
ax[2][2].scatter(x, roi1_TTP, c="g", s=4, label="roi1_TTP")
ax[2][2].scatter(x, roi2_TTP, c="r", s=4, label="roi2_TTP")
ax[2][2].legend()

ax[2][3].set_title('roi TTP diff')
ax[2][3].scatter(x, abs(np.array(roi1_TTP) - np.array(roi2_TTP)), c="g", s=4, label="roi_TTP_diff")

# target brightness
ax[0][0].set_title(f"target_TTP_diff1  {target_TTP_diff1_percent:.2f}")
ax[0][0].scatter(x, target_TTP_diff1, c="g", s=4)
ax[0][0].axhline(y=threshould_target, color='black', linestyle='-')

ax[0][1].set_title(f"target_TTP_diff2  {target_TTP_diff2_percent:.2f}")
ax[0][1].scatter(x, target_TTP_diff2, c="g", s=4)
ax[0][1].axhline(y=threshould_target, color='black', linestyle='-')

# target and cmp brightness diff
ax[1][0].set_title(f"diff_TTP_diff1  {diff_TTP_diff1_percent:.2f}")
ax[1][0].scatter(x, diff_TTP_diff1, c="b", s=4)
ax[1][0].axhline(y=threshould_diff, color='black', linestyle='-')

ax[1][1].set_title(f"diff_TTP_diff2  {diff_TTP_diff2_percent:.2f}")
ax[1][1].scatter(x, diff_TTP_diff2, c="b", s=4)
ax[1][1].axhline(y=threshould_diff, color='black', linestyle='-')

# contrast brightness
ax[2][0].set_title(f"contrast_TTP_diff1  {contrast_TTP_diff1_percent:.2f}")
ax[2][0].scatter(x, contrast_TTP_diff1, c="gray", s=4)
ax[2][0].axhline(y=threshould_contrast, color='black', linestyle='-')

ax[2][1].set_title(f"contrast_TTP_diff2  {contrast_TTP_diff2_percent:.2f}")
ax[2][1].scatter(x, contrast_TTP_diff2, c="gray", s=4)
ax[2][1].axhline(y=threshould_contrast, color='black', linestyle='-')

plt.suptitle(exp_path, y=0.92)
plt.savefig(os.path.join(save_path, f'{os.path.basename(exp_path)} metrics.png'))
plt.show()

# 分析数据

In [ ]:
def median_iqr(data):
    """
    Calculate the median and interquartile range (IQR) of a list of numbers.
    Returns a string formatted as 'median (Q1-Q3)'.
    """
    data = np.array(data)
    data = data[~np.isnan(data)]
    median = np.median(data)
    q1 = np.percentile(data, 25)
    q3 = np.percentile(data, 75)
    return f"{median:.2f} ({q1:.2f}-{q3:.2f})  min-max:{np.min(data):.2f}-{np.max(data):.2f}"

print("roi1_TTP", len(roi1_TTP), np.median(roi1_TTP), np.mean(roi1_TTP))
print("VueBox TTP", median_iqr(roi1_TTP))
print("Target TTP", median_iqr(target_TTP))
print("Diff TTP", median_iqr(diff_TTP))
print("Contrast TTP", median_iqr(contrast_TTP))
print("time_yolo_phase", median_iqr(time_yolo_phase))
print("time_analysis_phase", median_iqr(time_analysis_phase))
print("time_total", median_iqr(time_total))
print("ssim", median_iqr(ssim_list))
print("psnr", median_iqr(psnr_list))

threshould = 2
width = 0.5
t = [0, 0.5, 1.0, 1.5, 2.0, 2.5]
target_acc_list = []
diff_acc_list = []
contrast_acc_list = []

for i, threshould in enumerate(t):
    if i == len(t) - 1:
        width = 1000
        
    print(f"{threshould} <= x < {threshould + width}", end="    ")
         
    target_acc = len(
        [x for x in target_TTP_diff1 if threshould <= x < threshould + width]
    ) / len(target_TTP_diff1)

    diff_acc = len(
        [x for x in diff_TTP_diff1 if threshould <= x < threshould + width]
    ) / len(diff_TTP_diff1)

    contrast_acc = len(
        [x for x in contrast_TTP_diff1 if threshould <= x < threshould + width]
    ) / len(contrast_TTP_diff1)

    target_acc_list.append(target_acc)
    diff_acc_list.append(diff_acc)
    contrast_acc_list.append(contrast_acc)

name = ""
if exp_path.__contains__("qinzhou1"):
    name = "qinzhou1"
elif exp_path.__contains__("qinzhou2"):
    name = "qinzhou2"
elif exp_path.__contains__("hunan"):
    name = "hunan"
elif exp_path.__contains__("shihezi"):
    name = "shihezi"

print("\n\nAcc list Sum: ", sum(target_acc_list), sum(diff_acc_list), sum(contrast_acc_list))
print(f"target_acc_list_{name} = ", target_acc_list)
print(f"diff_acc_list_{name} = ", diff_acc_list)
print(f"contrast_acc_list_{name} = ", contrast_acc_list)
print(f'target_acc_{name} = ', sum(target_acc_list[:4]))
print(f'diff_acc_{name} = ', sum(diff_acc_list[:4]))
print(f'contrast_acc_{name} = ', sum(contrast_acc_list[:4]))